## PSO identifiability analysis

Loads the full PSO swarm history and asks which knobs actually constrained the inverse-design optimisation. The history is flattened to one row per candidate (knobs, cost, cost split, replicate variance, particle index). Because PSO is an optimiser, not a designed experiment (i.e., particles follow correlated trajectories toward the global best) a raw knob–cost correlation is confounded by that shared drift.

Two subsets are used. `sub` keeps every feasible candidate and gives the global best. `fit` also drops iteration 0, the uniform random initial swarm: it samples the whole search box far from where the swarm later works, and its large costs (up to ~8 in the runs of the accompanying article) would set the fitted curvature by leverage alone. Every statistic below is computed on `fit`.

The analysis follows the Methods of the accompanying article (PSO parameters identifiability):

- *Quadratic surface*: second-order polynomial in the standardised knobs. Drop-one partial $R^2$ gives the variance each knob explains uniquely.

- *Curvature*: eigendecomposition of the fitted Hessian. Large eigenvalue = stiff direction; near zero = sloppy; negative = the fitted cost falls toward a bound along it.

- *Bootstrap over particle trajectories*: particles resampled with replacement, each with its whole trajectory, the quadratic refitted 1000 times: a 95 % interval on each ranked eigenvalue and the fraction of resamples in which its dominant knob is unchanged. Particles interact through the global best, so the intervals are narrower than the true uncertainty.

- *Classification*: stiff, sloppy or boundary-limited per knob, identified or not per run.

By default the notebook reads the search of `use_cases/optimize_gaussian_psd.py`, which must be run first. That search (2 parameters, 4 particles, 3 iterations) only shows how the analysis runs: it is far too small for the verdicts to mean anything. For a real campaign, set `pso_file` and `config_file` in the configuration cell to the `pso_state.json` written by `python -m nanocarbon optimize` and to the configuration that produced it.

SPDX-License-Identifier: CC-BY-NC-SA-4.0  
Copyright (c) 2026 Gemmate Technologies s.r.l.

In [ ]:
# IMPORTS

import json
import numpy as np
import os
import runpy
import pandas as pd

from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures


In [ ]:
def load_pso_state(file_path: Path, save_csv: bool = False, csv_folder: Path | None = None, tag: str = ""):
    pso_state = json.load(open(file_path))
    ANCHOR_LABELS = pso_state['meta']['anchor_labels']
    ANCHOR_NUMBER = len(ANCHOR_LABELS)
    KNOB_LABELS = pso_state['meta']['param_labels']
    
    # UNPACK THE PSO HISTORY
    iteration = np.concatenate(
                [[i] * len(h['X']) for i, h in enumerate(pso_state['history'])]
                ).reshape(-1, 1)
    knobs = np.vstack(
            [np.array(h['X']) for h in pso_state['history']]
        ) # (iterations, # knobs)
    cost = np.vstack(
            [np.array(h['costs']).reshape(-1, 1) for h in pso_state['history'] ]
        ) # (iterations, 1)
    anchors = np.vstack([
                [np.nan]*ANCHOR_NUMBER if it is None else [it['anchors'][label] for label in ANCHOR_LABELS]
                for h in pso_state['history'] for it in h['extra']
            ])
    cost_var = np.array([
                np.nan if it is None else it['cost_var']
                for h in pso_state['history'] for it in h['extra']
            ]).reshape(-1, 1)
    num_survivors = np.array([
                    np.nan if it is None else it['n_survivors']
                    for h in pso_state['history'] for it in h['extra']
                ]).reshape(-1, 1)
    pso_df = pd.DataFrame(data= np.column_stack((iteration, knobs, cost, anchors, cost_var, num_survivors)),
                      columns= ["iter"]+KNOB_LABELS+["cost"]+ANCHOR_LABELS+["cost_var", "num_survivors"])
    pso_df['feasible'] = np.isfinite(pso_df['cost'])

    if save_csv and csv_folder is not None:
        new_csv_file = csv_folder / Path(tag + '.csv')
        if new_csv_file.exists(): new_csv_file.unlink() #remove the file if it's already there.
        pso_df.to_csv(new_csv_file, index=False)

    return pso_df, KNOB_LABELS, ANCHOR_LABELS

In [ ]:
def dump(df, name):
    f = EXPORT_DIR / (TAG + "__" + name + ".csv")
    if f.exists(): f.unlink()
    df.to_csv(f, index=False)
    return df


In [ ]:
def curvature(df: pd.DataFrame, knob_labels, cost_label = 'cost') -> tuple[np.ndarray, np.ndarray]:
    pf = PolynomialFeatures(degree=2,
                        interaction_only=False,
                        include_bias=True)
    X = df[knob_labels]
    y = df[cost_label]

    # scale
    sc = StandardScaler()
    X_transformed = sc.fit_transform(X = X)
    #polynomial transform
    X_transformed_polynomial = pf.fit_transform(X_transformed)
    polynomial_knobs_labels = pf.get_feature_names_out(knob_labels)
    #fit
    lr = LinearRegression()
    lr.fit(X_transformed_polynomial, y)
    beta = lr.coef_
    
    H = np.zeros((len(knob_labels), len(knob_labels))) #initialize hessian matrix

    for idx, label in enumerate(polynomial_knobs_labels):
        if label in ["1"]+knob_labels:
            continue # constant or 1st-degree coefficient, we can skip it
        for i, sup_string in enumerate(knob_labels):
            cleaned = label.replace("^2", "")
            if "^2" in label and cleaned == sup_string:
                H[i, i] = 2*beta[idx] # diagonal
                break
            for j, sub_string in enumerate(knob_labels):
                cleaned = label.replace(sup_string, "").replace(" ", "")
                if cleaned == sub_string.replace(" ", ""):
                    H[i, j] = beta[idx] # off-diagonal
                    break

    eigenvalues, eigenvectors = np.linalg.eigh(H)
    return eigenvalues, eigenvectors


In [ ]:
# CONFIG
path_to_notebook = Path(os.path.abspath(""))
# Defaults: the search of use_cases/optimize_gaussian_psd.py, run it first. For your own search, point
# pso_file to its pso_state.json and config_file to the configuration that defines its PARAM_BOUNDS.
example_dir = path_to_notebook.parent / "use_cases"
pso_file = example_dir / "output" / "calibration_gaussian" / "pso_state.json"
config_file = example_dir / "optimize_gaussian_psd.py"
EXPORT_DIR = pso_file.parent / "identifiability_csv"  # next to the search it analyses
SAVE_CSV = True
EXPORT_DIR.mkdir(exist_ok=True)
TAG = pso_file.stem  # carried as a `run` column
# Load PSO state and convrts the structure to a Pandas' DataFrame
pso_df, knob_labels, anchor_labels = load_pso_state(file_path = pso_file,
                                                    save_csv= SAVE_CSV, csv_folder= EXPORT_DIR, tag=TAG)
pso_df['particle'] = pso_df.groupby('iter').cumcount()  # 0..15 within each iteration
sub = pso_df[pso_df['feasible']] # all iterations: the global best is read from here

fit = sub[sub['iter'] > 0]

In [ ]:
# Quadratic surface: global quadratic cost fitting on polynomially expanded knobs 

pf = PolynomialFeatures(degree=2,
                        interaction_only=False,
                        include_bias=True)
X = fit[knob_labels]
y = fit['cost']
# scale
sc = StandardScaler()
X_transformed = sc.fit_transform(X = X)
#polynomial transform
X_transformed_polynomial = pf.fit_transform(X_transformed)
polynomial_knobs_labels = pf.get_feature_names_out(knob_labels)
#fit
lr = LinearRegression()
lr.fit(X_transformed_polynomial, y)
beta = lr.coef_
r2_poly = r2_score(y, lr.predict(X_transformed_polynomial))
print(f"R2 score, polynomial fitting: {r2_poly}")

# partial R2 coefficients -- polynomial fitting
r2_poly_drop={}
for label in knob_labels:
    partial = fit[knob_labels].drop(label, axis = 1)
    sc = StandardScaler()
    X_transformed_partial = sc.fit_transform(X = partial)
    #polynomial transform
    X_transformed_polynomial_partial = pf.fit_transform(X_transformed_partial)
    #fit
    lr = LinearRegression()
    lr.fit(X_transformed_polynomial_partial, y)
    beta_partial = lr.coef_
    r2_poly_drop[label] = r2_score(y, lr.predict(X_transformed_polynomial_partial))
dump(pd.DataFrame({"run":TAG, "knob":knob_labels,
                   "partial_r2_poly":[r2_poly - r2_poly_drop[k] for k in knob_labels]}), "r3_poly_regression")


In [ ]:
# Curvature: eigendecomposition of the Hessian of the quadratic fit.
# The Hessian is symmetric: real eigenvalues, orthonormal eigenvectors. A large eigenvalue is a stiff
# direction, and the largest components of its eigenvector say which knobs make it.
eigenvalues, eigenvectors = curvature(fit, knob_labels)

dump(pd.DataFrame({"run":TAG, "mode":range(len(eigenvalues)),
                   "eigenvalue":eigenvalues}), "r3_eigenvalues")

dump(pd.DataFrame(eigenvectors, index=knob_labels,
                  columns=[f"mode{m}" for m in range(len(eigenvalues))])
       .reset_index(names="knob")
       .melt(id_vars="knob", var_name="mode", value_name="loading")
       .assign(run=TAG), "r3_eigenvectors")

In [ ]:
# scalars → run, quad_r2, noise share eta (Methods), signal_fraction = 1 - eta, noise_floor
se2 = (fit['cost_var'] / fit['num_survivors']).mean()    # mean SE^2 on the replicate-mean cost
dump(pd.DataFrame({"run":[TAG], 
                   "quad_r2":[r2_poly],
                   "eta":[se2/fit['cost'].var()], # replicate-noise share of the cost variance
                   "signal_fraction":[1 - se2/fit['cost'].var()], # the R² ceiling
                   "noise_floor_se":[np.sqrt(se2)], # noise on the mean (ceiling scale, convergence band)
                   "noise_floor_struct":[np.sqrt(fit['cost_var'].mean())], # single-generation reproducibility
                   }), "fit_scores")


In [ ]:
# Bootstrap: BOOTSTRAPPING CURVATURE over particle trajectories.
# A resample draws the particles with replacement, each carrying its whole trajectory
# (iterations 1-25), so the dependence between successive positions of a particle is kept.
# The quadratic is refitted (and re-standardised) on every resample by curvature().
# Eigenvalues are ranked largest first. The order of two close eigenvalues can swap between
# resamples, so each ranked direction is also labelled by its dominant knob (largest |component|).
N_BOOT = 1000
rng = np.random.default_rng(0)  # fixed seed: the intervals are reproducible
by_particle = {p: g for p, g in fit.groupby('particle')}  # particle -> its trajectory rows
particles = list(by_particle)

eig_full, vec_full = curvature(fit, knob_labels)
eig_full, vec_full = eig_full[::-1], vec_full[:, ::-1]  # eigh sorts ascending: flip to largest first
dominant_full = np.array([knob_labels[k] for k in np.abs(vec_full).argmax(axis=0)])

eig_boot = np.zeros((N_BOOT, len(knob_labels)))  # (resamples, ranked directions)
dominant_boot = np.empty((N_BOOT, len(knob_labels)), dtype=object)
for b in range(N_BOOT):
    ids = rng.choice(particles, size=len(particles), replace=True)
    boot = pd.concat([by_particle[p] for p in ids])
    eig, vec = curvature(boot, knob_labels)
    eig_boot[b] = eig[::-1]
    dominant_boot[b] = [knob_labels[k] for k in np.abs(vec[:, ::-1]).argmax(axis=0)]

dump(pd.DataFrame({"run": TAG,
                   "rank": range(len(knob_labels)),
                   "eigenvalue": eig_full,
                   "ci_low": np.percentile(eig_boot, 2.5, axis=0),
                   "ci_high": np.percentile(eig_boot, 97.5, axis=0),
                   "dominant_knob": dominant_full,
                   "same_dominant_frac": (dominant_boot == dominant_full).mean(axis=0)}), "r3_bootstrap")

In [ ]:
# Classification (Methods): stiff, sloppy or boundary-limited per knob; identified or not per run.
# The thresholds are conventions of the paper, not statistical tests.
PARTIAL_MIN = 0.05  # a stiff knob explains at least this share of the cost variance uniquely
ROBUST_MIN = 0.85   # ... and keeps dominating its direction in at least this fraction of the resamples
BOUND_FRAC = 0.10   # a best value closer to a bound than this fraction of the range sits at the bound

bounds = {str(k): v for k, v in runpy.run_path(str(config_file))['PARAM_BOUNDS'].items()}
best = sub.loc[sub['cost'].idxmin(), knob_labels]           # the global best candidate
same_frac = (dominant_boot == dominant_full).mean(axis=0)   # per ranked direction, from the bootstrap

rows = []
for knob in knob_labels:
    lo, hi = bounds[knob]
    positive = (dominant_full == knob) & (eig_full > 0)      # positive-curvature directions this knob dominates
    stiff = r2_poly - r2_poly_drop[knob] >= PARTIAL_MIN and positive.any()
    dominance = same_frac[positive].max() if positive.any() else np.nan
    edge = min(best[knob] - lo, hi - best[knob]) / (hi - lo)  # distance from the nearer bound, fraction of the range
    if stiff:
        verdict = "stiff, robust" if dominance >= ROBUST_MIN else "stiff"
    else:
        verdict = "boundary-limited" if edge <= BOUND_FRAC else "sloppy"
    rows.append({"run": TAG, "knob": knob, "partial_r2": r2_poly - r2_poly_drop[knob],
                 "dominance": dominance, "best": best[knob], "edge": edge, "verdict": verdict})

classes = dump(pd.DataFrame(rows), "classification")
identified = (classes["verdict"] == "stiff, robust").any()
print(f"{TAG}: " + ("identified" if identified else "not identified, plateau of unidentifiability"))
classes